# Hybrid v2 — Vicsek Alignment + Anisotropic Noise + Speed Variation

**Changes from v1:**
- Speed varies per agent per step based on local alignment: `speed_i = V_min + (V_max - V_min) * local_pol_i`
- Agents in well-aligned neighborhoods move fast (hopping), agents in disordered neighborhoods move slow (walking)
- This is biologically motivated: field data shows speed is trimodal (0.2 / 2.7 / 11.8 cm/s) and correlated with local group order

**Expected improvements:**
- Turning angle std decreases (slow agents accumulate less angular displacement per step)
- NND increases (speed variation spreads agents out — fast agents pull ahead, slow agents fall behind)
- Forward void may emerge (fast aligned agents outrun neighbors, creating depletion ahead)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import json, os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

FIGURES_DIR = 'figures/'
FPS = 25
DT  = 1.0 / FPS

FIELD_POL_MEAN   = 0.8196
FIELD_POL_STD    = 0.0615
FIELD_NND_MEDIAN = 3.893
FIELD_NND_MEAN   = 4.473
FIELD_TA_STD     = 0.276

Lx = 99.44857450176137
Ly = 55.93982315724077

print("Ready.")

In [ ]:
def run_hybrid_v2(n_agents, Lx, Ly, v_min, v_max, eta_base, lambda_pull,
                  cone_half_angle, r_interaction, r_repulsion, dt, n_steps, seed=42):
    """
    Hybrid v2: Vicsek alignment + anisotropic noise + order-dependent speed.
    
    New vs v1:
    - Each agent's speed depends on local polarization of its neighborhood:
      speed_i = v_min + (v_max - v_min) * local_pol_i
    - Well-aligned neighborhoods → fast (hopping). Disordered → slow (walking).
    
    Parameters
    ----------
    v_min : float — speed when local_pol = 0 (walking mode, ~2.7 cm/s)
    v_max : float — speed when local_pol = 1 (hopping mode, ~11.8 cm/s)
    """
    rng = np.random.default_rng(seed)
    
    pos     = rng.uniform([0, 0], [Lx, Ly], size=(n_agents, 2))
    heading = rng.uniform(-np.pi, np.pi, size=n_agents)
    
    pos_history     = np.empty((n_steps + 1, n_agents, 2))
    heading_history = np.empty((n_steps + 1, n_agents))
    speed_history   = np.empty((n_steps + 1, n_agents))
    pos_history[0]     = pos
    heading_history[0] = heading
    speed_history[0]   = (v_min + v_max) / 2  # initial speed
    
    for step in range(n_steps):
        # Pairwise displacements with periodic BC
        delta = pos[np.newaxis, :, :] - pos[:, np.newaxis, :]
        delta[:, :, 0] -= Lx * np.round(delta[:, :, 0] / Lx)
        delta[:, :, 1] -= Ly * np.round(delta[:, :, 1] / Ly)
        dist = np.hypot(delta[:, :, 0], delta[:, :, 1])
        
        # --- Vicsek alignment ---
        align_mask = dist < r_interaction
        unit_vecs  = np.exp(1j * heading)
        neighbour_sum = align_mask @ unit_vecs
        mean_heading  = np.angle(neighbour_sum)
        
        # --- Local polarization (per agent) ---
        n_neighbors = align_mask.sum(axis=1).astype(float)
        n_neighbors = np.maximum(n_neighbors, 1)
        local_pol = np.abs(neighbour_sum) / n_neighbors  # [0, 1]
        
        # --- Order-dependent speed ---
        speed = v_min + (v_max - v_min) * local_pol
        
        # --- Forward fraction for anisotropic noise ---
        bearing_abs = np.arctan2(delta[:, :, 1], delta[:, :, 0])
        rel_bearing = bearing_abs - heading[:, np.newaxis]
        rel_bearing = (rel_bearing + np.pi) % (2 * np.pi) - np.pi
        
        in_range = (dist < r_interaction) & (dist > 0)
        in_cone  = in_range & (np.abs(rel_bearing) < cone_half_angle)
        
        n_in_range = in_range.sum(axis=1)
        n_in_cone  = in_cone.sum(axis=1)
        f_forward  = n_in_cone / np.maximum(n_in_range, 1)
        
        # --- Anisotropic noise ---
        eta_eff = eta_base * (1.0 - lambda_pull * f_forward)
        noise = rng.uniform(-0.5, 0.5, size=n_agents) * eta_eff
        
        new_heading = mean_heading + noise
        
        # --- Short-range repulsion ---
        rep_mask = (dist < r_repulsion) & (dist > 0)
        has_rep  = rep_mask.any(axis=1)
        if has_rep.any():
            rep_dx = -(rep_mask * delta[:, :, 0]).sum(axis=1)
            rep_dy = -(rep_mask * delta[:, :, 1]).sum(axis=1)
            rep_heading = np.arctan2(rep_dy, rep_dx)
            new_heading[has_rep] = rep_heading[has_rep] + noise[has_rep]
        
        heading = new_heading
        
        # --- Move at variable speed ---
        pos[:, 0] = (pos[:, 0] + speed * dt * np.cos(heading)) % Lx
        pos[:, 1] = (pos[:, 1] + speed * dt * np.sin(heading)) % Ly
        
        pos_history[step + 1]     = pos
        heading_history[step + 1] = heading
        speed_history[step + 1]   = speed
    
    return pos_history, heading_history, speed_history

print("Hybrid v2 model defined.")

In [ ]:
def polarization_from_headings(hh):
    return np.abs(np.exp(1j * hh).mean(axis=1))

def sim_turning_angles(hh):
    dtheta = np.diff(hh, axis=0)
    return ((dtheta + np.pi) % (2 * np.pi) - np.pi).ravel()

def sim_nnd(ph, Lx, Ly, subsample=20):
    nnds = []
    for t in range(0, len(ph), subsample):
        pos = ph[t]
        d = pos[np.newaxis,:,:] - pos[:,np.newaxis,:]
        d[:,:,0] -= Lx * np.round(d[:,:,0] / Lx)
        d[:,:,1] -= Ly * np.round(d[:,:,1] / Ly)
        dd = np.hypot(d[:,:,0], d[:,:,1])
        np.fill_diagonal(dd, np.inf)
        nnds.extend(np.min(dd, axis=1))
    return np.array(nnds)

def sim_neighbor_density_map(ph, hh, Lx, Ly, radius=10.0, nbins=50, subsample=5):
    edges = np.linspace(-radius, radius, nbins + 1)
    hist = np.zeros((nbins, nbins))
    for t in range(0, len(ph), subsample):
        pos, theta = ph[t], hh[t]
        d = pos[np.newaxis,:,:] - pos[:,np.newaxis,:]
        d[:,:,0] -= Lx * np.round(d[:,:,0] / Lx)
        d[:,:,1] -= Ly * np.round(d[:,:,1] / Ly)
        dd = np.hypot(d[:,:,0], d[:,:,1])
        for i in range(len(pos)):
            m = (dd[i] > 0.1) & (dd[i] < radius)
            if not m.any(): continue
            rel = d[i, m]
            r = np.pi/2 - theta[i]
            rx = rel[:,0]*np.cos(r) - rel[:,1]*np.sin(r)
            ry = rel[:,0]*np.sin(r) + rel[:,1]*np.cos(r)
            h, _, _ = np.histogram2d(rx, ry, bins=edges)
            hist += h
    return hist, edges

print("Metrics defined.")

## 2D Sweep: η_base × λ (with speed variation)

In [ ]:
SHARED = dict(
    n_agents=150, Lx=Lx, Ly=Ly,
    v_min=2.7,                      # walking speed from field data
    v_max=11.76,                    # hopping speed from field data
    r_interaction=7.0, r_repulsion=1.5,
    dt=DT, cone_half_angle=np.pi / 3,
)

BURN_IN = 500

eta_vals    = np.arange(0.3, 1.6, 0.1)
lambda_vals = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

pol_grid = np.zeros((len(eta_vals), len(lambda_vals)))

print(f"Sweeping {len(eta_vals)} x {len(lambda_vals)} = {len(eta_vals)*len(lambda_vals)} combinations...")
for i, eta in enumerate(eta_vals):
    for j, lam in enumerate(lambda_vals):
        _, hh, _ = run_hybrid_v2(**SHARED, eta_base=eta, lambda_pull=lam, n_steps=1500, seed=42)
        pol = polarization_from_headings(hh)[BURN_IN:]
        pol_grid[i, j] = pol.mean()
    print(f"  eta={eta:.1f}  pol = {['%.3f' % p for p in pol_grid[i]]}")

# Save sweep arrays
np.save('hybrid_eta_lambda_sweep.npy', pol_grid)
np.save('hybrid_eta_vals.npy', eta_vals)
np.save('hybrid_lambda_vals.npy', lambda_vals)

# Find best
err = np.abs(pol_grid - FIELD_POL_MEAN)
best_i, best_j = np.unravel_index(err.argmin(), err.shape)
best_eta = eta_vals[best_i]
best_lam = lambda_vals[best_j]
print(f"\nBest: eta_base={best_eta:.1f}, lambda={best_lam:.1f}  pol={pol_grid[best_i, best_j]:.3f}")

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pol_grid, origin='lower', aspect='auto',
               extent=[lambda_vals[0]-0.1, lambda_vals[-1]+0.1,
                       eta_vals[0]-0.05, eta_vals[-1]+0.05],
               cmap='RdYlBu_r', vmin=0.4, vmax=1.0)
ax.plot(best_lam, best_eta, 'k*', markersize=15, label=f'Best: η={best_eta:.1f}, λ={best_lam:.1f}')
cs = ax.contour(lambda_vals, eta_vals, pol_grid, levels=[FIELD_POL_MEAN],
                colors='black', linewidths=2, linestyles='--')
ax.clabel(cs, fmt=f'Φ={FIELD_POL_MEAN:.3f}')
ax.set(xlabel='λ (noise suppression)', ylabel='η_base (rad)',
       title=f'Hybrid v2: Polarization (v_min={SHARED["v_min"]}, v_max={SHARED["v_max"]})')
plt.colorbar(im, ax=ax, label='Steady-state polarization')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'hybrid_noise_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Final calibrated run + full metrics

In [ ]:
print(f"Running final hybrid v2: eta_base={best_eta:.1f}, lambda={best_lam:.1f}")
ph, hh, sh = run_hybrid_v2(**SHARED, eta_base=best_eta, lambda_pull=best_lam, n_steps=2000, seed=42)

pol_sim    = polarization_from_headings(hh)
pol_steady = pol_sim[BURN_IN:]
ta_sim     = sim_turning_angles(hh[BURN_IN:])

print(f"Polarization: mean={pol_steady.mean():.3f}  std={pol_steady.std():.3f}  (field: {FIELD_POL_MEAN:.3f})")
print(f"Turning angle std: {np.std(ta_sim):.3f}  (field: {FIELD_TA_STD:.3f})")

# Speed distribution
speeds_steady = sh[BURN_IN:].ravel()
print(f"Speed: mean={speeds_steady.mean():.2f}  std={speeds_steady.std():.2f} cm/s")
print(f"  (field: walking=2.7, hopping=11.8)")

print("\nComputing NND...")
nnd_sim = sim_nnd(ph[BURN_IN:], Lx, Ly)
print(f"NND: median={np.median(nnd_sim):.2f}  mean={nnd_sim.mean():.2f}  (field: {FIELD_NND_MEDIAN:.2f} / {FIELD_NND_MEAN:.2f})")

print("\nComputing neighbor density map...")
hist_sim, edges = sim_neighbor_density_map(ph[BURN_IN:], hh[BURN_IN:], Lx, Ly, radius=10.0, subsample=10)
print("Done.")

# --- Comparison table ---
print("\n" + "=" * 70)
print(f"{'Metric':<30} {'Field':>10} {'Vicsek':>10} {'Hybrid v1':>10} {'Hybrid v2':>10}")
print("=" * 70)
print(f"{'Polarization (mean)':<30} {FIELD_POL_MEAN:>10.3f} {'0.824':>10} {'0.812':>10} {pol_steady.mean():>10.3f}")
print(f"{'Turning angle std (rad)':<30} {FIELD_TA_STD:>10.3f} {'0.595':>10} {'0.622':>10} {np.std(ta_sim):>10.3f}")
print(f"{'NND median (cm)':<30} {FIELD_NND_MEDIAN:>10.2f} {'2.57':>10} {'2.47':>10} {np.median(nnd_sim):>10.2f}")
print(f"{'NND mean (cm)':<30} {FIELD_NND_MEAN:>10.2f} {'2.93':>10} {'2.75':>10} {nnd_sim.mean():>10.2f}")
print("=" * 70)

In [ ]:
# --- Neighbor density map ---
fig, ax = plt.subplots(figsize=(7, 6))
hist_norm = hist_sim / (hist_sim.sum() + 1e-10)
im = ax.imshow(hist_norm.T, origin='lower',
               extent=[edges[0], edges[-1], edges[0], edges[-1]],
               cmap='hot', aspect='equal')
ax.plot(0, 0, 'w^', markersize=12)
ax.axhline(0, color='white', ls='--', alpha=0.3)
ax.axvline(0, color='white', ls='--', alpha=0.3)
ax.set(xlabel='Left <- -> Right (cm)', ylabel='Behind <- -> Ahead (cm)',
       title=f'Hybrid v2 — Neighbor density\n(eta={best_eta:.1f}, lam={best_lam:.1f}, v=[{SHARED["v_min"]},{SHARED["v_max"]}])')
plt.colorbar(im, ax=ax, label='Relative density')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'hybrid_metric_3.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: hybrid_metric_3.png")

# --- Speed distribution ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(speeds_steady, bins=80, range=(0, 15), density=True, color='#27a060', alpha=0.7)
ax.axvline(SHARED['v_min'], color='blue', ls='--', lw=1.5, label=f'v_min={SHARED["v_min"]}')
ax.axvline(SHARED['v_max'], color='red', ls='--', lw=1.5, label=f'v_max={SHARED["v_max"]}')
ax.set(xlabel='Speed (cm/s)', ylabel='Density', title='Hybrid v2 — Speed distribution')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'hybrid_v2_speed_dist.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Export JSON ---
results = {
    "hybrid": {
        "n_agents": SHARED['n_agents'],
        "Lx": SHARED['Lx'], "Ly": SHARED['Ly'],
        "v_min": SHARED['v_min'], "v_max": SHARED['v_max'],
        "speed": float(speeds_steady.mean()),
        "eta_base": float(best_eta),
        "lambda_pull": float(best_lam),
        "cone_half_angle": float(SHARED['cone_half_angle']),
        "eta_empirical": 0.26,
        "eta_calibrated": float(best_eta),
        "r_interaction": SHARED['r_interaction'],
        "r_repulsion": SHARED['r_repulsion'],
        "dt": SHARED['dt'],
        "n_steps": 2000, "burn_in": BURN_IN,
    },
    "metrics": {
        "polarization_mean": float(pol_steady.mean()),
        "polarization_std": float(pol_steady.std()),
        "turning_angle_std": float(np.std(ta_sim)),
        "nnd_median": float(np.median(nnd_sim)),
        "nnd_mean": float(nnd_sim.mean()),
    },
    "field_targets": {
        "polarization_mean": FIELD_POL_MEAN, "polarization_std": FIELD_POL_STD,
        "turning_angle_std": FIELD_TA_STD,
        "nnd_median": FIELD_NND_MEDIAN, "nnd_mean": FIELD_NND_MEAN,
    }
}

with open('week2_hybrid_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved: week2_hybrid_results.json")

# Verify
print("\n--- Output verification ---")
for f in ['week2_hybrid_results.json', 'hybrid_eta_lambda_sweep.npy',
          'hybrid_eta_vals.npy', 'hybrid_lambda_vals.npy',
          'figures/hybrid_metric_3.png', 'figures/hybrid_noise_sweep.png']:
    print(f"  {f}: {'OK' if os.path.exists(f) else 'MISSING'}")